# 02 · Três métodos de seleção e votação das features

Regra fixada **antes** da comparação (`config.yaml → votacao`): cada método ranqueia as features candidatas
e vota nas suas **k = 10** melhores. Uma feature é eleita com **≥ 2 dos 3 votos**. Se sobrarem menos de **6**,
completa-se pela menor média de ranking, que também desempata e ordena.

Os três métodos são ajustados **só com o treino do ciclo**, numa amostra estratificada de 200 mil linhas:

| Método | O que mede |
|---|---|
| `SelectKBest(mutual_info_classif)` | dependência estatística (inclusive não linear) entre cada feature e o alvo |
| Importância de `RandomForestClassifier` | quanto cada feature reduz a impureza nas árvores |
| `RFE(LogisticRegression)` | ordem em que a feature é eliminada por ter o menor coeficiente |

In [ ]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [ ]:
from src import features as fs, preprocessamento as pp
ciclo = ler_ciclo(cfg)
treino = pd.read_parquet(estado(cfg, "processados") / "treino.parquet")
v = cfg["votacao"]
amostra = fs.amostrar(treino, v["amostra"], v["seed"])
print(ciclo["cycle_id"], ciclo["data_version"], "| amostra:", len(amostra), "linhas,",
      f"{amostra[pp.ALVO].mean():.2%} positivos | candidatas:", len(pp.CANDIDATAS))

In [ ]:
tabelas = {nome: metodo(amostra, pp.CANDIDATAS, v["k"], v["seed"]) for nome, metodo in fs.METODOS.items()}
pd.concat({n: t[["score", "ranking", "voto"]] for n, t in tabelas.items()}, axis=1).sort_values(("random_forest", "ranking")).round(4)

## Tabela de votos e features eleitas

In [ ]:
votacao = fs.votar(tabelas, v["min_votos"], v["min_features"])
eleitas = list(votacao.index[votacao["eleita"]])
print(f"{len(eleitas)} features eleitas:", eleitas)
votacao

In [ ]:
ev = pasta_evidencias(cfg, ciclo)
votacao.to_csv(ev / "votacao.csv")
mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · votação", tags=tags_ciclo(ciclo, etapa="votacao")) as run:
    mlflow.log_params({"metodos": ",".join(fs.METODOS), "k": v["k"], "min_votos": v["min_votos"],
                       "min_features": v["min_features"], "desempate": v["desempate"], "amostra": len(amostra),
                       "seed": v["seed"], "features_eleitas": ",".join(eleitas)})
    mlflow.log_metric("n_features_eleitas", len(eleitas))
    mlflow.log_artifact(ev / "votacao.csv")
    for nome, t in tabelas.items():
        t.to_csv(ev / f"ranking_{nome}.csv"); mlflow.log_artifact(ev / f"ranking_{nome}.csv", "rankings")
ciclo = atualizar_ciclo(cfg, features_eleitas=eleitas, run_votacao=run.info.run_id)